# Stage 2 — Combined Analysis

This is the centrepiece analysis notebook. It brings together safety and faithfulness results to make the causal argument.

## The causal argument

Stage 1 showed that certain experts are *correlated* with safe/faithful behaviour (high |RD|). Correlation alone is not causation — those experts could be passengers rather than drivers.

Stage 2 tests causation by suppressing those experts and observing whether behaviour changes predictably:

| Intervention | Expected if causal | Null expectation |
|---|---|---|
| Suppress compliance experts | safe_rate increases | no change |
| Suppress refusal experts | safe_rate decreases | no change |
| Suppress confabulation experts | faithfulness increases | no change |

If both directions of safety steering work (safe_rate increases *and* decreases depending on which experts are suppressed), this is strong bidirectional causal evidence.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid', font='serif')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

RESULTS_PATH = '/scratch/sc23jc3/stage2_results/results.json'
OUT_DIR = 'figures'
os.makedirs(OUT_DIR, exist_ok=True)

with open(RESULTS_PATH) as f:
    results = json.load(f)

safety = results['safety']
faith  = results['faithfulness']

## 1. Effect Size Overview — All Conditions

A single plot showing the delta (hard − baseline) for every condition. Positive = effect in the expected direction.

In [ ]:
def get_delta(d, key='hard'):
    b = d.get('baseline', float('nan'))
    h = d.get(key, float('nan'))
    if np.isnan(b) or np.isnan(h):
        return float('nan')
    return h - b

conditions = [
    ('Safety: suppress\ncompliance experts',  get_delta(safety['safe_steering']),   '#2980b9', True),
    ('Safety: suppress\nrefusal experts',     get_delta(safety['unsafe_steering']), '#e74c3c', False),
    ('Faithfulness:\nsuppress confabulation', get_delta(faith.get('counterfactual', {})), '#27ae60', True),
    ('Faithfulness:\nunanswerable',           get_delta(faith.get('unanswerable', {})),   '#f39c12', True),
    ('Control:\nSQuAD (should be flat)',      get_delta(faith.get('mctest', {})),          '#95a5a6', None),
]

labels = [c[0] for c in conditions]
deltas = [c[1] for c in conditions]
colors = [c[2] for c in conditions]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(len(labels)), deltas, color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=1.0)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Delta (Hard − Baseline)')
ax.set_title('Effect of Hard Expert Deactivation Across All Conditions')

for i, (d, (_, _, _, expected_positive)) in enumerate(zip(deltas, conditions)):
    if not np.isnan(d):
        sign = '+' if d >= 0 else ''
        ax.text(i, d + (0.005 if d >= 0 else -0.012), f'{sign}{d:.3f}',
                ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'combined_effect_sizes.png'), dpi=300, bbox_inches='tight')
plt.show()

## 2. Asymmetry Analysis — Safety

Does suppressing compliance experts raise safe_rate by the same magnitude that suppressing refusal experts lowers it? Symmetry would suggest the experts cleanly partition safe vs unsafe behaviour.

In [ ]:
delta_safe   = get_delta(safety['safe_steering'])
delta_unsafe = get_delta(safety['unsafe_steering'])

print(f'Safe steering delta:   {delta_safe:+.4f}  (suppress compliance experts → expect positive)')
print(f'Unsafe steering delta: {delta_unsafe:+.4f}  (suppress refusal experts → expect negative)')

if not (np.isnan(delta_safe) or np.isnan(delta_unsafe)):
    asymmetry = abs(delta_safe) - abs(delta_unsafe)
    print(f'\nAsymmetry (|safe delta| − |unsafe delta|): {asymmetry:+.4f}')
    if abs(asymmetry) < 0.05:
        print('  → Near-symmetric: experts contribute roughly equally in both directions.')
    elif asymmetry > 0:
        print('  → Suppressing compliance experts has a larger effect than suppressing refusal experts.')
    else:
        print('  → Suppressing refusal experts has a larger effect than suppressing compliance experts.')

## 3. Control Check

The SQuAD control score should remain flat. A large drop would suggest the steering degrades general capability rather than specifically targeting the intended behaviour.

In [ ]:
if 'mctest' in faith:
    b = faith['mctest'].get('baseline', float('nan'))
    h = faith['mctest'].get('hard',     float('nan'))
    d = h - b
    print(f'SQuAD control — Baseline: {b:.3f}, Hard: {h:.3f}, Delta: {d:+.3f}')
    if abs(d) < 0.05:
        print('  → Control is stable. Steering is specific to the targeted behaviour.')
    elif d < -0.05:
        print('  → Control drops significantly. Steering may be degrading general capability.')
    else:
        print('  → Control improves slightly (unexpected — worth investigating).')

## 4. Summary Table

In [ ]:
rows = [
    ('Safety: suppress compliance experts', safety['safe_steering'],           'positive'),
    ('Safety: suppress refusal experts',   safety['unsafe_steering'],          'negative'),
    ('Faithfulness: counterfactual',       faith.get('counterfactual', {}),    'positive'),
    ('Faithfulness: unanswerable',         faith.get('unanswerable', {}),      'positive'),
    ('Control: SQuAD',                     faith.get('mctest', {}),            'flat'),
]

print(f"{'Condition':40s} {'Baseline':>10} {'Hard':>10} {'Delta':>10} {'Expected':>10}")
print('-' * 80)
for label, row, expected in rows:
    b = row.get('baseline', float('nan'))
    h = row.get('hard',     float('nan'))
    d = h - b if not (np.isnan(b) or np.isnan(h)) else float('nan')
    print(f"{label:40s} {b:10.3f} {h:10.3f} {d:+10.3f} {expected:>10}")

## Discussion

*(Fill in after examining the results.)*

**Overall assessment**:
- Is there consistent evidence of causal expert involvement in both safety and faithfulness?
- Are the effects bidirectional for safety (both directions of steering work)?
- Is the control condition stable, validating that the effect is specific?

**Limitations**:
- n=100 prompts per condition — what would a full-scale run add?
- Hard deactivation is crude; soft suppression may show different dynamics
- Only CANDIDATE_N=10 experts per layer tested; is this the optimal N?

**Next steps**:
- Expand to full dataset sizes
- Introduce soft suppression and compare effect profiles
- Sweep CANDIDATE_N to find the optimal intersection size